# Build Your Own Retrieval Pipeline with Custom Data

**Goal:** Show that khoji works on any data — not just pre-packaged benchmarks. Build a retrieval system from scratch using data you control.

**Setup:**
- **Large model:** `BAAI/bge-base-en-v1.5` (110M) as the "off-the-shelf" baseline
- **Small model:** `sentence-transformers/all-MiniLM-L6-v2` (22M) to fine-tune
- **Dataset:** We'll load the MS MARCO subset from HuggingFace and convert it to khoji's format
- **Bonus:** Custom metrics, custom loss function, full training curve analysis

**Plan:**
1. Load external data and convert to khoji's `RetrievalDataset`
2. Baseline both models
3. Fine-tune MiniLM with three negative strategies
4. Add custom metrics (precision@k, hit_rate@k)
5. Plot and compare everything

In [ ]:
!pip install khoji matplotlib

## Step 1: Build a RetrievalDataset from Scratch

khoji's `RetrievalDataset` is just three dicts — queries, corpus, and relevance judgments. You can build it from any source: CSVs, databases, APIs, dataframes, or even hardcoded data.

Here we'll use SciFact but load it manually to show the pattern works for any data.

In [ ]:
import khoji

# Method 1: Load from BEIR (for reference)
scifact_beir = khoji.load_beir("scifact", split="train")
print(f"BEIR format: {len(scifact_beir.queries)} queries, {len(scifact_beir.corpus)} docs")

# Method 2: Build the exact same dataset manually from dicts
# This is what you'd do with your own data
my_dataset = khoji.RetrievalDataset(
    queries=scifact_beir.queries,   # dict: {query_id: query_text}
    corpus=scifact_beir.corpus,     # dict: {doc_id: doc_text}
    qrels=scifact_beir.qrels,       # dict: {query_id: {doc_id: relevance}}
)

# Peek at the data
sample_qid = list(my_dataset.queries.keys())[0]
sample_query = my_dataset.queries[sample_qid]
sample_rel_doc = list(my_dataset.qrels[sample_qid].keys())[0]

print(f"\nSample query ({sample_qid}): {sample_query[:100]}...")
print(f"Relevant doc ({sample_rel_doc}): {my_dataset.corpus[sample_rel_doc][:100]}...")

In [ ]:
# Your own data might come from anywhere. Here's the pattern:
#
# import pandas as pd
#
# questions = pd.read_csv("questions.csv")  # columns: id, text
# articles = pd.read_csv("articles.csv")    # columns: id, content
# labels = pd.read_csv("labels.csv")        # columns: question_id, article_id, score
#
# dataset = khoji.RetrievalDataset(
#     queries={str(r.id): r.text for r in questions.itertuples()},
#     corpus={str(r.id): r.content for r in articles.itertuples()},
#     qrels={
#         str(qid): {str(aid): int(s) for _, aid, s in grp.itertuples()}
#         for qid, grp in labels.groupby("question_id")
#     },
# )

## Step 2: Baseline Evaluation with Custom Metrics

khoji comes with nDCG, MRR, and Recall. But you can add any metric you want.

In [ ]:
# Define custom metrics
def precision_at_k(ranked_doc_ids, qrel, k):
    """Fraction of top-k results that are relevant."""
    relevant = {d for d, s in qrel.items() if s > 0}
    found = sum(1 for d in ranked_doc_ids[:k] if d in relevant)
    return found / k

def hit_rate_at_k(ranked_doc_ids, qrel, k):
    """1 if any relevant doc appears in top-k, else 0."""
    relevant = {d for d, s in qrel.items() if s > 0}
    return 1.0 if any(d in relevant for d in ranked_doc_ids[:k]) else 0.0

extra_metrics = {
    "precision": precision_at_k,
    "hit_rate": hit_rate_at_k,
}

In [ ]:
# Load test set
test_dataset = khoji.load_beir("scifact", split="test")

# BGE baseline (110M)
bge_eval = khoji.Evaluator("BAAI/bge-base-en-v1.5")
bge_baseline = bge_eval.evaluate(
    dataset_name="scifact",
    k_values=[1, 5, 10],
    dataset=test_dataset,
    extra_metrics=extra_metrics,
)
bge_baseline.print()
del bge_eval

In [ ]:
# MiniLM baseline (22M)
minilm_eval = khoji.Evaluator("sentence-transformers/all-MiniLM-L6-v2")
minilm_baseline = minilm_eval.evaluate(
    dataset_name="scifact",
    k_values=[1, 5, 10],
    dataset=test_dataset,
    extra_metrics=extra_metrics,
)
minilm_baseline.print()
del minilm_eval

## Step 3: Fine-tune MiniLM — Three Strategies, Pure Python API

No YAML — everything programmatic. Build triplets, configure training, run.

In [ ]:
from functools import partial
from khoji.loss import infonce_loss

MODEL = "sentence-transformers/all-MiniLM-L6-v2"
LORA = khoji.LoRASettings(r=16, alpha=32, dropout=0.1)
LOSS_FN = partial(infonce_loss, temperature=0.05)

def make_config(save_dir):
    return khoji.TrainingConfig(
        epochs=5,
        batch_size=16,
        lr=2e-5,
        warmup_steps=50,
        max_length=512,
        loss_fn=LOSS_FN,
        lora=LORA,
        save_dir=save_dir,
        sanity_check_samples=5,
    )

def evaluate_adapter(adapter_path):
    ev = khoji.Evaluator(MODEL, adapter_path=adapter_path)
    result = ev.evaluate(
        dataset_name="scifact",
        k_values=[1, 5, 10],
        dataset=test_dataset,
        extra_metrics=extra_metrics,
    )
    del ev
    return result

In [ ]:
# --- Strategy 1: Random negatives ---
random_triplets = khoji.build_random_negatives(my_dataset, n_negatives=3)
print(f"Random: {len(random_triplets)} triplets")

trainer = khoji.Trainer(MODEL, make_config("./output/custom-random/adapter"))
history_random = trainer.train(khoji.TripletDataset(random_triplets))
result_random = evaluate_adapter("./output/custom-random/adapter")
result_random.print()

In [ ]:
# --- Strategy 2: Hard negatives with skip_top ---
# skip_top=5 avoids the top 5 most similar non-relevant docs (often false negatives)
mining_model = khoji.EmbeddingModel(MODEL)
hard_triplets = khoji.mine_hard_negatives(
    my_dataset, mining_model, n_negatives=3, top_k=50,
    skip_top=5,  # <-- skip top 5 likely mislabeled negatives
)
del mining_model
print(f"Hard (skip_top=5): {len(hard_triplets)} triplets")

trainer = khoji.Trainer(MODEL, make_config("./output/custom-hard/adapter"))
history_hard = trainer.train(khoji.TripletDataset(hard_triplets))
result_hard = evaluate_adapter("./output/custom-hard/adapter")
result_hard.print()

In [ ]:
# --- Strategy 3: Mixed negatives ---
mining_model = khoji.EmbeddingModel(MODEL)
mixed_triplets = khoji.build_mixed_negatives(
    my_dataset, mining_model, n_random=2, n_hard=1, top_k=50,
)
del mining_model
print(f"Mixed: {len(mixed_triplets)} triplets")

trainer = khoji.Trainer(MODEL, make_config("./output/custom-mixed/adapter"))
history_mixed = trainer.train(khoji.TripletDataset(mixed_triplets))
result_mixed = evaluate_adapter("./output/custom-mixed/adapter")
result_mixed.print()

## Step 4: Compare Everything

In [ ]:
all_results = {
    "BGE (110M)": bge_baseline.metrics,
    "MiniLM base": minilm_baseline.metrics,
    "+ random": result_random.metrics,
    "+ hard": result_hard.metrics,
    "+ mixed": result_mixed.metrics,
}

# Focus metrics
focus = ["ndcg@10", "mrr@10", "recall@10", "precision@10", "hit_rate@10"]

header = f"{'Metric':<15}" + "".join(f"{name:>14}" for name in all_results)
print(header)
print("=" * len(header))
for metric in focus:
    row = f"{metric:<15}"
    for name, metrics in all_results.items():
        val = metrics.get(metric, 0)
        row += f"{val:>14.4f}"
    print(row)

# Gap analysis
bge_ndcg = bge_baseline.metrics["ndcg@10"]
base_ndcg = minilm_baseline.metrics["ndcg@10"]
best_ndcg = max(r.metrics["ndcg@10"] for r in [result_random, result_hard, result_mixed])
gap_before = bge_ndcg - base_ndcg
gap_after = bge_ndcg - best_ndcg

print(f"\nnDCG@10 gap vs BGE: {gap_before:.4f} → {gap_after:.4f}")
if gap_before > 0:
    print(f"Gap closed by {100 * (1 - gap_after / gap_before):.1f}%")

## Step 5: Training Curve Analysis

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

histories = {
    "Random": history_random,
    "Hard": history_hard,
    "Mixed": history_mixed,
}
colors = {"Random": "#2196F3", "Hard": "#FF5722", "Mixed": "#4CAF50"}

# Per-step loss
for name, hist in histories.items():
    axes[0, 0].plot(hist.step_loss, label=name, alpha=0.7, color=colors[name])
axes[0, 0].set_title("Loss per Optimizer Step")
axes[0, 0].set_xlabel("Step")
axes[0, 0].legend()

# Per-epoch loss
for name, hist in histories.items():
    axes[0, 1].plot(hist.epoch_loss, marker="o", label=name, color=colors[name])
axes[0, 1].set_title("Average Loss per Epoch")
axes[0, 1].set_xlabel("Epoch")
axes[0, 1].legend()

# Learning rate
for name, hist in histories.items():
    axes[1, 0].plot(hist.step_lr, label=name, alpha=0.7, color=colors[name])
axes[1, 0].set_title("Learning Rate Schedule")
axes[1, 0].set_xlabel("Step")
axes[1, 0].legend()

# Gradient norm
for name, hist in histories.items():
    axes[1, 1].plot(hist.step_grad_norm, label=name, alpha=0.5, color=colors[name])
axes[1, 1].set_title("Gradient Norm per Step")
axes[1, 1].set_xlabel("Step")
axes[1, 1].legend()

plt.suptitle("Training Dynamics: Random vs Hard vs Mixed Negatives", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("custom_training_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

## Step 6: Inference with the Fine-Tuned Model

In [ ]:
import torch

# Load the best model
model = khoji.EmbeddingModel(MODEL, adapter_path="./output/custom-mixed/adapter")

# Your queries
queries = [
    "Does caffeine consumption affect sleep quality?",
    "Is exercise effective for treating depression?",
]

# Your corpus (in practice, this would be your knowledge base)
corpus = [
    "Caffeine intake within 6 hours of bedtime significantly reduces total sleep time.",
    "Regular physical activity has been shown to reduce symptoms of depression.",
    "Vitamin C supplementation does not prevent the common cold.",
    "Moderate exercise improves mood through endorphin release.",
    "High caffeine doses can disrupt circadian rhythm and REM sleep.",
    "Omega-3 fatty acids may have mild antidepressant effects.",
]

q_embs = model.encode(queries)
c_embs = model.encode(corpus)
scores = torch.mm(q_embs, c_embs.t())

for i, query in enumerate(queries):
    ranked = torch.argsort(scores[i], descending=True).tolist()
    print(f"\nQuery: {query}")
    for rank, idx in enumerate(ranked[:3]):
        print(f"  {rank+1}. (score={scores[i][idx]:.3f}) {corpus[idx]}")

## Step 7: Iterative Mining Rounds via Python API

The `run()` pipeline handles mining rounds automatically via YAML. But you can also do it manually with the Python API for full control — useful when you want different hyperparameters per round.

In [ ]:
## Summary

### What we showed

1. **Any data works** — `RetrievalDataset` is just three dicts. Load from CSV, database, API, or build manually.
2. **Custom metrics** — plug in `precision@k`, `hit_rate@k`, or any function with the right signature.
3. **Three negative strategies** — random (fast), hard (precise), mixed (balanced). All via Python API.
4. **`skip_top`** — skip the most similar non-relevant docs that are likely false negatives. Critical for datasets with incomplete labels.
5. **Iterative mining rounds** — manually control each round: mine → train → re-mine with fine-tuned model → train again with lower LR.
6. **Small model + fine-tuning matches large model** — 22M params with LoRA approaches 110M params.
7. **Full observability** — training curves, gradient norms, LR schedules, sanity checks.

### The composable toolkit

Every component is independent:
- `RetrievalDataset` — your data in, three dicts
- `build_random_negatives` / `mine_hard_negatives` / `build_mixed_negatives` — triplet construction with `skip_top` support
- `Trainer` — training loop with LoRA, AMP, gradient clipping, `adapter_path` for warm-start
- `Evaluator` — retrieval metrics with custom metric support
- `EmbeddingModel` — inference with optional adapter loading

Use the full `run()` pipeline with `mining_rounds` in YAML for convenience, or pick individual pieces for custom workflows.

## Summary

### What we showed

1. **Any data works** — `RetrievalDataset` is just three dicts. Load from CSV, database, API, or build manually.
2. **Custom metrics** — plug in `precision@k`, `hit_rate@k`, or any function with the right signature.
3. **Three negative strategies** — random (fast), hard (precise), mixed (balanced). All via Python API.
4. **Small model + fine-tuning matches large model** — 22M params with LoRA approaches 110M params.
5. **Full observability** — training curves, gradient norms, LR schedules, sanity checks.

### The composable toolkit

Every component is independent:
- `RetrievalDataset` — your data in, three dicts
- `build_random_negatives` / `mine_hard_negatives` / `build_mixed_negatives` — triplet construction
- `Trainer` — training loop with LoRA, AMP, gradient clipping
- `Evaluator` — retrieval metrics with custom metric support
- `EmbeddingModel` — inference with optional adapter loading

Use the full `run()` pipeline for convenience, or pick individual pieces for custom workflows.

In [ ]:
# Evaluate the 2-round model
result_2rounds = evaluate_adapter("./output/custom-rounds/adapter_final")

# Final comparison including mining rounds
print(f"{'Strategy':<30} {'nDCG@10':>10} {'MRR@10':>10} {'Recall@10':>10}")
print("-" * 62)
for name, metrics in [
    ("BGE (110M) baseline", bge_baseline.metrics),
    ("MiniLM baseline", minilm_baseline.metrics),
    ("+ random", result_random.metrics),
    ("+ hard (skip_top=5)", result_hard.metrics),
    ("+ mixed", result_mixed.metrics),
    ("+ mixed 2-round (skip_top=5)", result_2rounds.metrics),
]:
    print(f"{name:<30} {metrics['ndcg@10']:>10.4f} {metrics['mrr@10']:>10.4f} {metrics['recall@10']:>10.4f}")